Лендмарки на руке

In [1]:
import cv2
import numpy as np
from huggingface_hub import hf_hub_download
from ultralytics import YOLO


# Скачиваем готовую модель
model_path = hf_hub_download(
    repo_id="poptoz/yolo26-hand-pose-face-detection",
    filename="checkpoints/yolo26_hand_pose.pt",
)

model = YOLO(model_path)

input_path = "input.MOV"
output_path = "output_hand.avi"

cap = cv2.VideoCapture(input_path)

if not cap.isOpened():
    raise FileNotFoundError(
        f"Не удалось открыть видео: {input_path}"
    )

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

print("Разрешение:", width, "x", height)
print("FPS:", fps)

# AVI + MJPG обычно сохраняется стабильнее
writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"MJPG"),
    fps,
    (width, height),
)

if not writer.isOpened():
    raise RuntimeError("Не удалось создать выходное видео")


connections = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
    (5, 9), (9, 13), (13, 17),
]


def calculate_angle(point_a, point_b, point_c):
    a = np.array(point_a, dtype=np.float32)
    b = np.array(point_b, dtype=np.float32)
    c = np.array(point_c, dtype=np.float32)

    vector_1 = a - b
    vector_2 = c - b

    length_1 = np.linalg.norm(vector_1)
    length_2 = np.linalg.norm(vector_2)

    if length_1 == 0 or length_2 == 0:
        return 0

    cosine = np.dot(vector_1, vector_2) / (
        length_1 * length_2
    )

    cosine = np.clip(cosine, -1.0, 1.0)

    return np.degrees(np.arccos(cosine))


def count_fingers(points):
    fingers = 0

    finger_landmarks = [
        (5, 6, 8),
        (9, 10, 12),
        (13, 14, 16),
        (17, 18, 20),
    ]

    for mcp, pip, tip in finger_landmarks:
        angle = calculate_angle(
            points[mcp],
            points[pip],
            points[tip],
        )

        if angle > 150:
            fingers += 1

    thumb_angle = calculate_angle(
        points[1],
        points[2],
        points[4],
    )

    if thumb_angle > 145:
        fingers += 1

    return fingers


frame_number = 0

try:
    while True:
        success, frame = cap.read()

        if not success:
            break

        frame_number += 1

        # Пока запускаем на CPU для проверки стабильности
        results = model.predict(
            frame,
            device="cpu",
            imgsz=640,
            conf=0.3,
            verbose=False,
        )

        output = np.zeros_like(frame)

        total_fingers = 0

        for result in results:
            if result.keypoints is None:
                continue

            keypoints = result.keypoints.xy.cpu().numpy()

            for hand in keypoints:
                if len(hand) != 21:
                    continue

                points = []
                valid_hand = True

                for x, y in hand:
                    if x <= 0 or y <= 0:
                        valid_hand = False
                        break

                    points.append(
                        (int(x), int(y))
                    )

                if not valid_hand:
                    continue

                total_fingers += count_fingers(points)

                for start, end in connections:
                    cv2.line(
                        output,
                        points[start],
                        points[end],
                        (255, 255, 255),
                        2,
                    )

                for point in points:
                    cv2.circle(
                        output,
                        point,
                        5,
                        (255, 255, 255),
                        -1,
                    )

        cv2.putText(
            output,
            f"Fingers: {total_fingers}",
            (30, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.5,
            (255, 255, 255),
            3,
        )

        writer.write(output)

        if frame_number % 50 == 0:
            print("Обработано кадров:", frame_number)

finally:
    # Это выполнится даже при обычной Python-ошибке
    cap.release()
    writer.release()


print("Готово")
print("Кадров обработано:", frame_number)
print("Файл:", output_path)

/home/mlguest/miniforge3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Разрешение: 720 x 1280
FPS: 29.963144963144963
Обработано кадров: 50
Обработано кадров: 100
Обработано кадров: 150
Обработано кадров: 200
Обработано кадров: 250
Обработано кадров: 300
Обработано кадров: 350
Обработано кадров: 400
Обработано кадров: 450
Обработано кадров: 500
Обработано кадров: 550
Обработано кадров: 600
Обработано кадров: 650
Обработано кадров: 700
Обработано кадров: 750
Обработано кадров: 800
Готово
Кадров обработано: 813
Файл: output_hand.avi


Сегментация

In [1]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO


# Сегментационная модель
model = YOLO("yolo11n-seg.pt")

device = 0 if torch.cuda.is_available() else "cpu"

print("Устройство:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


input_path = "input1.mp4"
output_path = "output_segmentation.mp4"

cap = cv2.VideoCapture(input_path)

if not cap.isOpened():
    raise FileNotFoundError(
        f"Не удалось открыть видео: {input_path}"
    )

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

if not writer.isOpened():
    raise RuntimeError("Не удалось создать выходное видео")


# Набор цветов для разных объектов
colors = [
    (255, 0, 0),
    (0, 255, 0),
    (0, 0, 255),
    (255, 255, 0),
    (255, 0, 255),
    (0, 255, 255),
    (200, 100, 0),
    (100, 0, 200),
]


def check_interaction(mask1, mask2, kernel_size=7, min_intersection=50):
    # Немного расширяем маски, чтобы ловить касание границ
    kernel = np.ones((kernel_size, kernel_size), np.uint8)

    mask1_dilated = cv2.dilate(
        mask1.astype(np.uint8),
        kernel,
        iterations=1,
    )

    intersection = (
        (mask1_dilated > 0)
        & (mask2 > 0)
    )

    intersection_area = np.sum(intersection)

    return intersection_area > min_intersection


frame_number = 0

while True:
    success, frame = cap.read()

    if not success:
        break

    frame_number += 1

    # Запускаем сегментацию
    results = model.predict(
        frame,
        device=device,
        imgsz=640,
        conf=0.35,
        verbose=False,
        retina_masks=True,
    )

    output = frame.copy()
    interaction_messages = []

    for result in results:
        if result.masks is None or result.boxes is None:
            continue

        masks = result.masks.data.cpu().numpy()
        boxes = result.boxes
        classes = boxes.cls.cpu().numpy().astype(int)
        names = result.names

        binary_masks = []

        # Сначала подготавливаем маски
        for i, mask in enumerate(masks):
            mask_binary = (mask > 0.5).astype(np.uint8)

            # Если размер маски не совпадает с кадром, подгоняем
            if mask_binary.shape[0] != height or mask_binary.shape[1] != width:
                mask_binary = cv2.resize(
                    mask_binary,
                    (width, height),
                    interpolation=cv2.INTER_NEAREST,
                )

            binary_masks.append(mask_binary)

        # Рисуем маски и подписи
        for i, mask_binary in enumerate(binary_masks):
            color = colors[i % len(colors)]

            colored_mask = np.zeros_like(output)
            colored_mask[:, :, 0] = mask_binary * color[0]
            colored_mask[:, :, 1] = mask_binary * color[1]
            colored_mask[:, :, 2] = mask_binary * color[2]

            output = cv2.addWeighted(
                output,
                1.0,
                colored_mask,
                0.35,
                0,
            )

            # Находим контур маски
            contours, _ = cv2.findContours(
                mask_binary,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE,
            )

            cv2.drawContours(
                output,
                contours,
                -1,
                color,
                2,
            )

            # Подпись объекта
            ys, xs = np.where(mask_binary > 0)
            if len(xs) > 0 and len(ys) > 0:
                x_min = int(np.min(xs))
                y_min = int(np.min(ys))

                class_name = names[classes[i]]

                cv2.putText(
                    output,
                    class_name,
                    (x_min, max(y_min - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    color,
                    2,
                )

        # Проверяем взаимодействие между всеми парами объектов
        for i in range(len(binary_masks)):
            for j in range(i + 1, len(binary_masks)):
                mask1 = binary_masks[i]
                mask2 = binary_masks[j]

                if check_interaction(mask1, mask2):
                    class_1 = names[classes[i]]
                    class_2 = names[classes[j]]

                    message = f"{class_1} touches {class_2}"

                    interaction_messages.append(message)

                    print(
                        f"Кадр {frame_number}: {message}"
                    )

    # Убираем дубли сообщений
    interaction_messages = list(dict.fromkeys(interaction_messages))

    # Выводим сообщения на кадр
    if interaction_messages:
        cv2.putText(
            output,
            "Interaction detected",
            (30, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 0, 255),
            3,
        )

        for idx, message in enumerate(interaction_messages[:5]):
            cv2.putText(
                output,
                message,
                (30, 80 + idx * 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2,
            )

    writer.write(output)

    if frame_number % 50 == 0:
        print("Обработано кадров:", frame_number)


cap.release()
writer.release()

print("Готово")
print("Обработано кадров:", frame_number)
print("Результат:", output_path)

Устройство: 0
GPU: NVIDIA GeForce RTX 5070 Ti
Кадр 1: truck touches car
Кадр 2: car touches car
Кадр 2: person touches person
Кадр 3: car touches car
Кадр 3: person touches person
Кадр 4: car touches car
Кадр 4: person touches person
Кадр 6: car touches truck
Кадр 7: car touches person
Кадр 8: car touches person
Кадр 11: car touches car
Кадр 12: car touches car
Кадр 18: car touches truck
Кадр 19: car touches truck
Кадр 19: car touches truck
Кадр 19: truck touches truck
Кадр 23: car touches bus
Кадр 25: person touches person
Кадр 26: person touches person
Кадр 27: person touches person
Кадр 27: person touches person
Кадр 27: truck touches car
Кадр 28: person touches person
Кадр 28: truck touches car
Кадр 28: truck touches bus
Кадр 28: car touches bus
Кадр 29: person touches person
Кадр 30: bus touches car
Кадр 31: bus touches car
Кадр 31: person touches person
Кадр 32: bus touches car
Кадр 32: person touches person
Кадр 32: person touches person
Кадр 33: bus touches car
Кадр 34: bus tou